# ✈️ AeroLex — Building a RAG Pipeline for answering Delta's passenger's queries

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ grounded answer + citations  (+ long-term profile)        [#7,#9]
```


## 0 · Setup

In [ ]:
import os
import time
import numpy as np
from pathlib import Path
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.embeddings import GeminiEmbedder
from aerolex.generate import Generator
from aerolex.pipeline import RagPipeline
from aerolex.store import InMemoryVectorStore
from aerolex.chunking import chunk_corpus, load_chunks

In [26]:
# Set API key, LLM, Embedding Model 
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
GEN_MODEL = "gemini-3.5-flash-lite"
KEY = config.load_api_key()
CACHE_DIR = r"C:\src\IEEEWorkshop\aerolex-production-rag\cache"

## 1 · Hierarchical Chunking  ·  Feature #1

Legal text is a **hierarchy** (Article / § / RULE → paragraph). We split on those
*logical* boundaries — never fixed-size windows — and attach citation metadata, so
an answer can cite *"EU 261/2004 — Article 7"* instead of *"chunk 42"*.

In [27]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    chunks = load_chunks()
    print(f"\nloaded {len(chunks)} chunks from cache")
except FileNotFoundError:
    chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(chunks)} chunks")

by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

delta_domestic   delta   78,663 chars  Delta Domestic General Rules Tariff (Contract of Carriage)
delta_intl       delta  118,099 chars  Delta International General Rules Tariff (Contract of Carriage)
eu261            eu      27,544 chars  EU Regulation (EC) No 261/2004
cfr260           cfr     26,287 chars  US DOT 14 CFR Part 260

loaded 234 chunks from cache

Example chunk — EU Regulation (EC) No 261/2004 — Article 7
1.
Where reference is made to this Article, passengers shall
receive compensation amounting to:
(a) EUR 250 for all flights of 1 500 kilometres or less;
(b) EUR 400 for all intra-Community flights of more than
1 500 kilometres, and for all other flights between 1 500
and 3 500 kilometres;
(c) EUR 600 for all flights no …


# 3 · Embedding Creation

In [ ]:
if not KEY:
    print("GEMINI_API_KEY not set.")

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} "
        f"(dim={EMBED_DIM}) …")

t0 = time.time()
embedder = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM)
matrix = embedder.embed_documents([c.searchable_text for c in chunks])
print(f"Embedded in {time.time() - t0:.1f}s, shape={matrix.shape}")

Embedding 234 chunks with gemini-embedding-001 (dim=768) …


In [ ]:
store = InMemoryVectorStore([c.chunk_id for c in chunks], matrix)
store.save(Path(CACHE_DIR) / "new_embeddings.npz")

## 2 · Lexical Search with BM25  ·  Feature #2

Dense search blurs exact tokens, but statute codes must match **exactly**
(`§ 260.6`, `261/2004`). BM25 is a sparse bag-of-words scorer — the trick is a
**tokenizer that preserves those references** as single tokens.

### 🛠️ TODO: write `tokenize`

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Keep words and dotted/slashed numeric refs as single tokens.
_TOKEN = re.compile(r"[a-z]+|\d+(?:[./-]\d+)*")


def tokenize(text: str) -> list[str]:
    # TODO Fetch list of keywords
    tokens = re.findall(_TOKEN, text.lower())
    return [token for token in tokens if token not in ENGLISH_STOP_WORDS]

assert tokenize("What is the refund for a cancelled flight?") == ["refund", "cancelled", "flight"]
print("tokenize OK")

tokenize OK


In [ ]:
from aerolex.bm25 import BM25Index

bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])
for cid, score in bm25.search("§ 260.6 refund for a cancelled flight", top_k=3):
    print(f"{score:6.2f}  {by_id[cid].citation}")

 16.02  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
 15.18  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
 15.00  US DOT 14 CFR Part 260 — § 260.6 (part 3/6)


## 3 · Vector Search + Memory

Gemini embeds every chunk into a vector; the whole corpus is one NumPy matrix.
Similarity is just a **normalized dot product** (cosine).


### 🛠️ TODO: write `cosine_search` 

In [ ]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # TODO: 1) L2-normalize query_vec and each row of matrix
    q_hat = query_vec / np.sqrt(np.sum(query_vec ** 2))
    matrix_hat = matrix / np.sqrt(np.sum(matrix ** 2, axis=1, keepdims=True))
    #       2) scores = matrix @ query   (cosine, since normalized)
    scores = matrix_hat @ q_hat
    #       3) return the top_k (id, score) pairs, highest first
    top_k = np.argsort(-scores)[:top_k]
    top_chunks = [(ids[i], scores[i]) for i in top_k]
    return top_chunks

# sanity: a row is most similar to itself (cosine ≈ 1.0)
_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=1)[0]
print(_hit[0] == store.ids[0], round(_hit[1], 4))

True 1.0


In [ ]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)

dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=5)
for cid, s in dense_hits:
    print(f"{s:5.3f}  {by_id[cid].citation}")

0.726  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.724  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.713  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.698  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)
0.696  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)


## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine scores live on different scales, so we fuse by **rank**:
each list contributes `1 / (k + rank)` to an id's score (`k = 60`). Documents that
rank well in *either* retriever bubble up.

### 🛠️ TODO: write `rrf`

In [ ]:
def rrf(rankings, k=60, top_k=10):
    scores = {}
    for ranked in rankings:
        for rank, cid in enumerate(ranked):
            chunk_score = scores.get(cid, 0.0)
            # TODO for each chunk compute the RRF Score
            scores[cid] = chunk_score + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
print("rrf OK")

rrf OK


In [ ]:
bm25_ids = [cid for cid, _ in bm25.search(query, top_k=20)]
dense_ids = [cid for cid, _ in dense_hits]
fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=10)
print("Fused top-5:")
for cid, s in fused[:5]:
    print(f"{s:.4f}  {by_id[cid].citation}")

Fused top-5:
0.0323  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.0303  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.0296  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.0161  US DOT 14 CFR Part 260 — § 260.1
0.0159  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)


## 5 · Cross-Encoder Re-Ranking  ·  Feature #6

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [ ]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
final_chunks = [by_id[cid] for cid, _ in reranked]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranked top chunks:
   6.68  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
   6.28  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
   5.77  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)
   5.55  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
   5.28  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
   5.24  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
   3.78  US DOT 14 CFR Part 260 — § 260.1
   1.71  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)
   0.84  Delta International General Rules Tariff (Contract of Carriage) — RULE 15 (part 4/4)
  -0.09  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)


## 6 · Grounded Answer with Inline Citations  ·  Feature #7

The model answers **only** from retrieved passages and cites their chunk IDs.
Generation returns plain text without resolving or validating citation markers.
The numeric `resolve_citations` exercise below is optional and is not called by
the generator or full pipeline.

Edit `system_prompt` and `user_prompt_template` in the generation cell below.
The template inserts `{query}`, `{context_text}` (passages labeled by chunk ID), and
`{profile_block}` (optional user facts). Use `{{` and `}}` for literal braces.
Keep the context and `[[chunk_id]]` instructions for grounded answers. Omitting the
prompt overrides preserves the defaults; offline answers remain extractive.

### 🛠️ TODO: write `resolve_citations`

In [ ]:
# CITE = re.compile(r"\[\[(\d+)\]\]")

# def resolve_citations(answer, index):
#     used = []
#     def keep(m):
#         n = int(m.group(1))
#         if n in index:
#             if n not in used:
#                 used.append(n)
#             return m.group(0)
#         return ""                      # drop a citation to a non-existent passage
#     cleaned = CITE.sub(keep, answer)
#     return cleaned, [f"[[{n}]] {index[n]}" for n in sorted(used)]

# _c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
# print(_c, "|", _r)

In [ ]:
# TODO: Write a system prompt for the AeroLex assistant
system_prompt = (
    "You are AeroLex, an assistant answering questions about airline passenger "
    "rights using ONLY the provided context passages. Every factual sentence "
    "must end with a citation tag like [[chunk_id]] using the exact chunk_id of the passage "
    "you used. If the context does not answer the question, say so plainly. "
    "Never invent regulations, article numbers, or passage numbers."
)

In [ ]:
user_prompt_template = (
    "{profile_block}\n\n"
    "Context passages:\n{context_text}\n\n"
    "Question: {query}\n\n"
    "Answer (cite every factual claim with [[chunk_id]]):"
)

result = Generator(api_key = KEY, model=GEN_MODEL).generate(
    query, chunks,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt_template,
)

print(result)

When a flight is cancelled, the refund provisions depend on the specific governing terms and regulations:

* **Delta Domestic Rules:** If a refund is required due to a flight cancellation pursuant to Rule 19(A), passengers will receive a direct refund [[delta_domestic::rule-22::0]]. If no portion of the ticket has been used, the refund equals the fare paid [[delta_domestic::rule-22::0]]. If a portion of the ticket has been used, refunds are calculated based on whether the interruption occurs at a fare breakpoint or within a fare component, provided alternate transportation is not supplied and accepted [[delta_domestic::rule-22::0, delta_domestic::rule-22::1]]. Additionally, under Rule 19(A), if a flight is cancelled, Delta will cancel the remaining ticket and refund the unused portion along with unused ancillary fees (including paid checked bag fees) in the original form of payment [[delta_domestic::rule-19::0]].
* **US DOT Regulations (14 CFR Part 260):** For a cancelled flight, a cov

## 7 · Conversational Memory 

Follow-ups like *"what about in the US?"* are meaningless to a retriever alone. We
**condense** the chat history + follow-up into a standalone query before searching.

In [ ]:
from aerolex.memory import ConversationMemory

mem = ConversationMemory()
mem.add_user(query)
mem.add_assistant(result)
print("follow-up : what about in the US?")
print("standalone:", mem.condense("what about in the US?"))

follow-up : what about in the US?
standalone: Delta domestic and US DOT flight cancellation refund rules and regulations


## 8 · Long-Term Persistence 

Facts a user states persist across sessions in SQLite and can be injected into
future prompts.

In [ ]:
from aerolex.persistence import ProfileStore

prof = ProfileStore()
prof.remember("demo_user", "home_airport", "JFK")
prof.remember("demo_user", "status", "Delta Platinum Medallion")
print(prof.context_block("demo_user"))

Known facts about the user:
- home_airport: JFK
- status: Delta Platinum Medallion


## 10 · The Full Pipeline

Now assemble every stage with the `aerolex` library — the productionized version of
what you just built — and ask a question end to end.

In [ ]:
pipe = RagPipeline(
    chunks=chunks,
    store=store,
    bm25=bm25,
    embedder=GeminiEmbedder(),
    reranker=CrossEncoderReranker(),
    generator=Generator(),
)
out = pipe.answer("If my 4000 km flight is cancelled, what am I owed?",
                    memory=mem, profile=prof, user_id="demo_user",
                    system_prompt=system_prompt,
                    user_prompt_template=user_prompt_template)
print(out["answer"])

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

AttributeError: 'str' object has no attribute 'update'